In [17]:
using LowLevelFEM, LinearAlgebra, SparseArrays

In [18]:
macro peakrss(ex)
    quote
        GC.gc(true)

        local rss_before = Sys.maxrss()
        local t0 = time_ns()

        local value = $(esc(ex))

        local elapsed = (time_ns() - t0) / 1e9
        local rss_after = Sys.maxrss()

        println("Elapsed time: ",
            round(elapsed, digits=3), " s")
        println("Process peak RSS: ",
            round(rss_after / 2.0^30, digits=3), " GiB")
        println("New RSS high-water mark: ",
            round((rss_after - rss_before) / 2.0^20,
                digits=1), " MiB")

        value
    end
end

@peakrss (macro with 1 method)

In [19]:
structured_rect_mesh(x0=10.0, n=100, order=2)

In [20]:
mat = Material("body")
Pu = Problem([mat], type=:VectorField, dim=2, field=:u)

E = mat.E
ν = mat.ν

r = ScalarField(Pu, "body", (x, y, z)->x)
A1 = [1 0 0; 0 0 0; 0 1 0; 0 0 1]
A2 = [0 0; 1/r 0; 0 0; 0 0]
B = A1 ⋅ SymGrad(Pu) + A2 ⋅ Pu
D = E / (1+ν) / (1-2ν) * [1-ν ν ν 0; ν 1-ν ν 0; ν ν 1-ν 0; 0 0 0 (1-2ν)/2]

#@time K_original = ∫(B' ⋅ D ⋅ B * (2π*r), multithread=false)
expr = B' ⋅ D ⋅ B
#K_original[:, :]

LowLevelFEM.CompoundBilinear(LowLevelFEM.TransposedChain(LowLevelFEM.ChainSum(LowLevelFEM.ChainSumTerm[LowLevelFEM.ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.SymGradOp()), Any[[1 0 0 0; 0 0 1 0; 0 0 0 1]]), :full), LowLevelFEM.ChainSumTerm(LowLevelFEM.MatrixChain(LowLevelFEM.OpApplied(Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 40401, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :rhs), LowLevelFEM.IdOp()), Any[Any[0 ScalarField([[0.1; 0.0999000999000999; … ; 0.1; 0.09995

In [29]:
@time K_csc = ∫(
    expr * (2π*r);
    Ω="body",
    assembly=:ijv,
    threads=:auto
)

  1.532767 seconds (14.53 M allocations: 1.190 GiB, 18.71% gc time)


sparse([1, 2, 9, 10, 207, 208, 1399, 1400, 1599, 1600  …  1003, 1004, 21201, 21202, 80401, 80402, 80797, 80798, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [6.7668566204325585e6, 3.020681613254934e6, 376104.95304940373, -201464.69811888214, -5.264342493157097e6, 805697.6851598285, -1.1009134905033798e6, 201249.88836456763, 912807.3491419237, -804999.5534586333  …  -2148.097540036208, -2.4563065760865178e7, -5.902972042435402e6, -4.249581365189442e6, 8.090231453650176e-7, -946022.1570238082, 2148.097537484529, -2.4563065760866243e7, -1.6422135615812294e-6, 6.80207974916265e7], 80802, 80802)

In [31]:
@time K_csc = ∫(
    expr * (2π*r);
    Ω="body",
    assembly=:csc,
    threads=:auto
)

  9.225407 seconds (305.82 M allocations: 5.848 GiB, 10.61% gc time)


sparse([1, 2, 9, 10, 207, 208, 1399, 1400, 1599, 1600  …  1003, 1004, 21201, 21202, 80401, 80402, 80797, 80798, 80801, 80802], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802, 80802], [6.7668566204325585e6, 3.020681613254934e6, 376104.95304940373, -201464.69811888214, -5.264342493157097e6, 805697.6851598285, -1.1009134905033798e6, 201249.88836456763, 912807.3491419237, -804999.5534586333  …  -2148.097540036208, -2.4563065760865178e7, -5.902972042435402e6, -4.249581365189442e6, 8.092559760086715e-7, -946022.1570238082, 2148.097537484529, -2.456306576086624e7, -1.6422135615812294e-6, 6.80207974916265e7], 80802, 80802)

In [23]:
K_csc[:, :]

80802×80802 SparseMatrixCSC{Float64, Int64} with 2566318 stored entries:
⎡⣿⣿⡛⠛⠛⠛⠛⠛⠛⠛⠻⢿⣟⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠓⠶⢤⣄⣀⠀⠀⎥
⎢⣿⣆⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠙⠛⎥
⎢⣿⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⢷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠘⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⣿⠀⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦